In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", None)

df_train = pd.read_csv("train.csv/train.csv")
df_test = pd.read_csv("test.csv/test.csv")
df_test_labels = pd.read_csv("test_labels.csv/test_labels.csv")
df_sample_submission = pd.read_csv("sample_submission.csv/sample_submission.csv")

df_train.head()

In [ ]:
import html
import re

def preprocess_text(text):
    """Clean a comment while keeping wording, negation, and punctuation."""
    if pd.isna(text):
        return ""

    text = html.unescape(str(text))
    text = re.sub(r"https?://\S+|www\.\S+", " URL ", text)
    text = re.sub(r"\b[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}\b", " EMAIL ", text)
    text = re.sub(r"\b(?:\d{1,3}\.){3}\d{1,3}\b", " IPADDRESS ", text)
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text

# Keep the original text; use clean_text as the model's text input.
df_train["clean_text"] = df_train["comment_text"].apply(preprocess_text)
df_test["clean_text"] = df_test["comment_text"].apply(preprocess_text)

df_train[["comment_text", "clean_text"]].head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

# Select the comment text as the input features for the model.
X = df_train["clean_text"]

# Select the binary toxicity label that the model will learn to predict.
# A value of 0 means non-toxic, and a value of 1 means toxic.
y = df_train["toxic"]

# Split the labeled data into a training set and a validation set.
# The model learns from 80% of the comments and is evaluated on the other 20%.
# A fixed random_state makes the split reproducible.
# Stratification keeps the toxic/non-toxic proportions similar in both sets.
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Create a Bag of Words vectorizer that counts words in each comment.
# It learns the vocabulary only from training comments to prevent data leakage.
bow_vectorizer = CountVectorizer()

# Learn the vocabulary and convert each training comment into a word-count vector.
X_train_bow = bow_vectorizer.fit_transform(X_train)

# Convert validation comments using the same vocabulary learned from training.
X_valid_bow = bow_vectorizer.transform(X_valid)

# Show the number of comments and vocabulary terms in each matrix.
print("Training matrix:", X_train_bow.shape)
print("Validation matrix:", X_valid_bow.shape)

# Report how many unique terms the vectorizer learned from training comments.
print("Vocabulary size:", len(bow_vectorizer.get_feature_names_out()))

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix

# Train a Naive Bayes classifier using the Bag of Words training matrix.
bow_model = MultinomialNB()
bow_model.fit(X_train_bow, y_train)

# Predict the toxicity labels for comments kept out of training.
y_valid_pred = bow_model.predict(X_valid_bow)

# Report precision, recall, and F1-score for both classes.
print(classification_report(
    y_valid,
    y_valid_pred,
    target_names=["Non-toxic", "Toxic"],
    zero_division=0
))

# Rows are the actual labels and columns are the predicted labels.
print("Confusion matrix (rows = actual, columns = predicted):")
print(confusion_matrix(y_valid, y_valid_pred, labels=[0, 1]))

## Exploratory Data Analysis (EDA)

Before diving deeper into modeling, let's understand the data distribution
and class imbalance in our dataset.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ── Label distribution across all toxicity categories ──
label_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
label_counts = df_train[label_cols].sum().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: count of positive samples per label
axes[0].bar(label_counts.index, label_counts.values, color="#e74c3c")
axes[0].set_title("Positive samples per toxicity category")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=30)
for i, v in enumerate(label_counts.values):
    axes[0].text(i, v + 200, str(v), ha="center", fontsize=9)

# Pie chart: toxic vs non-toxic for the main "toxic" column
toxic_counts = df_train["toxic"].value_counts()
axes[1].pie(
    toxic_counts.values,
    labels=["Non-toxic", "Toxic"],
    autopct="%1.1f%%",
    colors=["#2ecc71", "#e74c3c"],
    startangle=90,
    explode=(0, 0.05)
)
axes[1].set_title("Toxic vs Non-toxic distribution")

plt.tight_layout()
plt.show()

print(f"Total comments: {len(df_train):,}")
print(f"Toxic comments: {df_train['toxic'].sum():,} ({df_train['toxic'].mean()*100:.1f}%)")
print(f"Clean comments: {(df_train['toxic']==0).sum():,} ({(1-df_train['toxic'].mean())*100:.1f}%)")

In [ ]:
# ── Comment length distribution: toxic vs non-toxic ──
df_train["comment_length"] = df_train["clean_text"].str.len()

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(
    df_train.loc[df_train["toxic"] == 0, "comment_length"],
    bins=100, alpha=0.6, label="Non-toxic", color="#2ecc71", density=True
)
ax.hist(
    df_train.loc[df_train["toxic"] == 1, "comment_length"],
    bins=100, alpha=0.6, label="Toxic", color="#e74c3c", density=True
)
ax.set_xlim(0, 2000)
ax.set_xlabel("Comment length (characters)")
ax.set_ylabel("Density")
ax.set_title("Comment length distribution by class")
ax.legend()
plt.tight_layout()
plt.show()

print("Mean comment length:")
print(df_train.groupby("toxic")["comment_length"].mean().rename({0: "Non-toxic", 1: "Toxic"}))

In [ ]:
# ── Label co-occurrence heatmap ──
corr = df_train[label_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, cmap="Reds", vmin=0, vmax=1)
ax.set_xticks(range(len(label_cols)))
ax.set_yticks(range(len(label_cols)))
ax.set_xticklabels(label_cols, rotation=45, ha="right")
ax.set_yticklabels(label_cols)
for i in range(len(label_cols)):
    for j in range(len(label_cols)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=9)
plt.colorbar(im, ax=ax)
ax.set_title("Label correlation matrix")
plt.tight_layout()
plt.show()

## Model Comparison: Naive Bayes vs Logistic Regression

Logistic Regression often outperforms Naive Bayes on text classification.
Let's compare both using the same Bag of Words features.

In [ ]:
from sklearn.linear_model import LogisticRegression

# Train a Logistic Regression classifier on the same BoW features.
# max_iter is increased because the default 100 may not converge on large vocabularies.
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_bow, y_train)

# Predict on the validation set.
y_valid_pred_lr = lr_model.predict(X_valid_bow)

print("=" * 60)
print("Logistic Regression — Bag of Words")
print("=" * 60)
print(classification_report(
    y_valid,
    y_valid_pred_lr,
    target_names=["Non-toxic", "Toxic"],
    zero_division=0
))
print("Confusion matrix (rows = actual, columns = predicted):")
print(confusion_matrix(y_valid, y_valid_pred_lr, labels=[0, 1]))

In [ ]:
from sklearn.metrics import f1_score, accuracy_score, recall_score, precision_score

# ── Side-by-side comparison of both models ──
models = {
    "Naive Bayes": y_valid_pred,
    "Logistic Regression": y_valid_pred_lr
}

metrics_data = []
for name, preds in models.items():
    metrics_data.append({
        "Model": name,
        "Accuracy": accuracy_score(y_valid, preds),
        "Precision (Toxic)": precision_score(y_valid, preds),
        "Recall (Toxic)": recall_score(y_valid, preds),
        "F1 (Toxic)": f1_score(y_valid, preds)
    })

df_metrics = pd.DataFrame(metrics_data).set_index("Model")
print(df_metrics.round(4).to_string())

# Bar chart comparison
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(df_metrics.columns))
width = 0.3
for i, (model_name, row) in enumerate(df_metrics.iterrows()):
    ax.bar(x + i * width, row.values, width, label=model_name)

ax.set_xticks(x + width / 2)
ax.set_xticklabels(df_metrics.columns, rotation=15)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.set_title("Model Comparison — Bag of Words")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

# ── Visual confusion matrices side by side ──
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (name, preds) in zip(axes, models.items()):
    ConfusionMatrixDisplay.from_predictions(
        y_valid, preds,
        display_labels=["Non-toxic", "Toxic"],
        cmap="Blues",
        ax=ax
    )
    ax.set_title(f"{name}")

plt.suptitle("Confusion Matrices — Bag of Words", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()